In [ ]:
import pandas as pd
import numpy as np

nav = pd.read_csv('data/processed/nav_history_clean.csv', parse_dates=['date'])
nav = nav.sort_values(['amfi_code', 'date'])

# Pivot: dates as rows, schemes as columns
nav_pivot = nav.pivot(index='date', columns='amfi_code', values='nav').sort_index()

# Daily returns
daily_returns = nav_pivot.pct_change().dropna(how='all')

# CAGR helper
def cagr(nav_series, years):
    nav_series = nav_series.dropna()
    if len(nav_series) < 2:
        return np.nan
    start, end = nav_series.iloc[0], nav_series.iloc[-1]
    n = years
    return (end / start) ** (1 / n) - 1

cagr_rows = []
for code in nav_pivot.columns:
    series = nav_pivot[code].dropna()
    cagr_rows.append({
        'amfi_code': code,
        'CAGR_1yr': cagr(series[series.index >= series.index.max() - pd.DateOffset(years=1)], 1),
        'CAGR_3yr': cagr(series[series.index >= series.index.max() - pd.DateOffset(years=3)], 3),
        'CAGR_5yr': cagr(series[series.index >= series.index.max() - pd.DateOffset(years=5)], 5),
    })

cagr_table = pd.DataFrame(cagr_rows)
print(nav_pivot.shape, daily_returns.shape)
cagr_table.head(10)

In [ ]:
import os
print("Current working directory:", os.getcwd())
print(os.listdir())

In [ ]:
import pandas as pd
import numpy as np

nav = pd.read_csv('../data/processed/nav_history_clean.csv', parse_dates=['date'])
nav = nav.sort_values(['amfi_code', 'date'])

nav_pivot = nav.pivot(index='date', columns='amfi_code', values='nav').sort_index()
daily_returns = nav_pivot.pct_change().dropna(how='all')

def cagr(nav_series, years):
    nav_series = nav_series.dropna()
    if len(nav_series) < 2:
        return np.nan
    start, end = nav_series.iloc[0], nav_series.iloc[-1]
    return (end / start) ** (1 / years) - 1

cagr_rows = []
for code in nav_pivot.columns:
    series = nav_pivot[code].dropna()
    cagr_rows.append({
        'amfi_code': code,
        'CAGR_1yr': cagr(series[series.index >= series.index.max() - pd.DateOffset(years=1)], 1),
        'CAGR_3yr': cagr(series[series.index >= series.index.max() - pd.DateOffset(years=3)], 3),
        'CAGR_5yr': cagr(series[series.index >= series.index.max() - pd.DateOffset(years=5)], 5),
    })

cagr_table = pd.DataFrame(cagr_rows)
print(nav_pivot.shape, daily_returns.shape)
cagr_table.head(10)

In [ ]:
RISK_FREE_RATE = 0.065  # RBI repo rate proxy, annualised
daily_rf = RISK_FREE_RATE / 252

sharpe_sortino_rows = []

for code in daily_returns.columns:
    returns = daily_returns[code].dropna()
    if len(returns) < 2:
        continue

    excess_returns = returns - daily_rf

    # Sharpe Ratio
    sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(252)

    # Sortino Ratio (denominator = downside deviation only, negative returns)
    downside_returns = excess_returns[excess_returns < 0]
    downside_std = downside_returns.std()
    sortino = (excess_returns.mean() / downside_std) * np.sqrt(252) if downside_std > 0 else np.nan

    sharpe_sortino_rows.append({
        'amfi_code': code,
        'Sharpe_Ratio': sharpe,
        'Sortino_Ratio': sortino,
    })

sharpe_sortino_table = pd.DataFrame(sharpe_sortino_rows)
sharpe_sortino_table = sharpe_sortino_table.sort_values('Sharpe_Ratio', ascending=False)
print(sharpe_sortino_table.shape)
sharpe_sortino_table.head(10)

In [ ]:
import os

print("--- data/raw ---")
print(os.listdir('../data/raw'))

print("\n--- data/processed ---")
print(os.listdir('../data/processed'))

In [ ]:
benchmark = pd.read_csv('../data/raw/10_benchmark_indices.csv')
print(benchmark.columns.tolist())
print(benchmark.head(5))
print(benchmark.shape)

In [ ]:
from scipy import stats

# Pivot benchmark: dates as rows, index names as columns
benchmark = pd.read_csv('../data/raw/10_benchmark_indices.csv', parse_dates=['date'])
benchmark_pivot = benchmark.pivot(index='date', columns='index_name', values='close_value').sort_index()
print(benchmark_pivot.columns.tolist())  # confirm exact names, e.g. NIFTY50 / NIFTY100

benchmark_returns = benchmark_pivot.pct_change().dropna(how='all')

# Align benchmark with fund daily_returns on common dates
common_dates = daily_returns.index.intersection(benchmark_returns.index)
fund_returns_aligned = daily_returns.loc[common_dates]
nifty100_returns = benchmark_returns.loc[common_dates, 'NIFTY100']  # adjust name if different

alpha_beta_rows = []
for code in fund_returns_aligned.columns:
    y = fund_returns_aligned[code].dropna()
    x = nifty100_returns.loc[y.index].dropna()
    y = y.loc[x.index]  # ensure same index after dropna

    if len(x) < 30:
        continue

    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    alpha_beta_rows.append({
        'amfi_code': code,
        'Alpha': intercept * 252,
        'Beta': slope,
        'R_squared': r_value ** 2,
    })

alpha_beta_table = pd.DataFrame(alpha_beta_rows).sort_values('Alpha', ascending=False)
print(alpha_beta_table.shape)
alpha_beta_table.head(10)

In [ ]:
print("nav index dtype:", daily_returns.index.dtype)
print("benchmark index dtype:", benchmark_returns.index.dtype)
print("nav date range:", daily_returns.index.min(), "to", daily_returns.index.max())
print("benchmark date range:", benchmark_returns.index.min(), "to", benchmark_returns.index.max())
print("duplicate amfi_codes in daily_returns columns?", daily_returns.columns.duplicated().sum())
print("total unique amfi_codes:", daily_returns.columns.nunique())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(daily_returns.index, daily_returns[119551].cumsum(), label='SBI Bluechip (cum. return)')
ax.plot(nifty100_returns.index, nifty100_returns.cumsum(), label='Nifty100 (cum. return)')
ax.legend()
plt.show()

In [ ]:
print("alpha_beta_table shape:", alpha_beta_table.shape)
print("duplicated amfi_codes:", alpha_beta_table['amfi_code'].duplicated().sum())

In [ ]:
max_dd_rows = []

for code in nav_pivot.columns:
    series = nav_pivot[code].dropna()
    running_max = series.cummax()
    drawdown = (series / running_max) - 1
    max_drawdown = drawdown.min()
    max_dd_rows.append({
        'amfi_code': code,
        'Max_Drawdown': max_drawdown,
    })

max_dd_table = pd.DataFrame(max_dd_rows).sort_values('Max_Drawdown')
print(max_dd_table.shape)
max_dd_table.head(10)

In [ ]:
# Merge all metrics into one master table
scorecard = cagr_table.merge(sharpe_sortino_table, on='amfi_code') \
                       .merge(alpha_beta_table, on='amfi_code') \
                       .merge(max_dd_table, on='amfi_code')

# Pull expense_ratio_pct from the reference file to use in scoring
expense_ref = pd.read_csv('../data/processed/scheme_performance_clean.csv')[['amfi_code', 'expense_ratio_pct']]
scorecard = scorecard.merge(expense_ref, on='amfi_code')

print(scorecard.shape)
scorecard.head()

In [ ]:
def to_percentile_rank(series, ascending=True):
    return series.rank(pct=True, ascending=ascending) * 100

scorecard['rank_3yr_return']  = to_percentile_rank(scorecard['CAGR_3yr'], ascending=True)
scorecard['rank_sharpe']      = to_percentile_rank(scorecard['Sharpe_Ratio'], ascending=True)
scorecard['rank_alpha']       = to_percentile_rank(scorecard['Alpha'], ascending=True)
scorecard['rank_expense']     = to_percentile_rank(scorecard['expense_ratio_pct'], ascending=False)  # lower expense = better
scorecard['rank_max_dd']      = to_percentile_rank(scorecard['Max_Drawdown'], ascending=True)  # less negative = better

scorecard['Fund_Score'] = (
    0.30 * scorecard['rank_3yr_return'] +
    0.25 * scorecard['rank_sharpe'] +
    0.20 * scorecard['rank_alpha'] +
    0.15 * scorecard['rank_expense'] +
    0.10 * scorecard['rank_max_dd']
)

scorecard = scorecard.sort_values('Fund_Score', ascending=False)
scorecard.to_csv('fund_scorecard.csv', index=False)
print(scorecard.shape)
scorecard[['amfi_code', 'Fund_Score', 'CAGR_3yr', 'Sharpe_Ratio', 'Alpha', 'expense_ratio_pct', 'Max_Drawdown']].head(10)

In [ ]:
top5_codes = scorecard.head(5)['amfi_code'].tolist()
three_yr_start = daily_returns.index.max() - pd.DateOffset(years=3)

fig, ax = plt.subplots(figsize=(12, 6))

for code in top5_codes:
    series = daily_returns.loc[daily_returns.index >= three_yr_start, code].dropna()
    ax.plot(series.index, series.cumsum(), label=f'Fund {code}')

nifty50_3yr = benchmark_returns.loc[benchmark_returns.index >= three_yr_start, 'NIFTY50'].dropna()
nifty100_3yr = benchmark_returns.loc[benchmark_returns.index >= three_yr_start, 'NIFTY100'].dropna()

ax.plot(nifty50_3yr.index, nifty50_3yr.cumsum(), label='Nifty50', linewidth=2, linestyle='--', color='black')
ax.plot(nifty100_3yr.index, nifty100_3yr.cumsum(), label='Nifty100', linewidth=2, linestyle=':', color='grey')

ax.set_title('Top 5 Funds vs Nifty50 / Nifty100 — 3 Year Cumulative Return')
ax.legend()
plt.tight_layout()
plt.savefig('benchmark_comparison_chart.png', dpi=150)
plt.show()

# Tracking error for each top-5 fund vs Nifty100
tracking_errors = []
for code in top5_codes:
    fund_ret = daily_returns.loc[daily_returns.index >= three_yr_start, code].dropna()
    common = fund_ret.index.intersection(nifty100_3yr.index)
    te = (fund_ret.loc[common] - nifty100_3yr.loc[common]).std() * np.sqrt(252)
    tracking_errors.append({'amfi_code': code, 'Tracking_Error': te})

pd.DataFrame(tracking_errors)

In [ ]:
alpha_beta_table.to_csv('alpha_beta.csv', index=False)
print("Saved alpha_beta.csv")

In [1]:
import pandas as pd
for f in ['03_aum_by_fund_house', '04_monthly_sip_inflows', '10_benchmark_indices']:
    df = pd.read_csv(f'../data/raw/{f}.csv')
    print(f, ":", df.columns.tolist())

03_aum_by_fund_house : ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
04_monthly_sip_inflows : ['month', 'sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']
10_benchmark_indices : ['date', 'index_name', 'close_value']


In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/05_category_inflows.csv')
print(df.columns.tolist())
print(df.head(5))


['month', 'category', 'net_inflow_crore']
     month         category  net_inflow_crore
0  2024-04        Large Cap            2413.0
1  2024-04          Mid Cap            3897.0
2  2024-04        Small Cap            3533.0
3  2024-04        Flexi Cap            4947.0
4  2024-04  Large & Mid Cap            4214.0


In [2]:
import os
print(os.listdir('../data/raw'))

['01_fund_master.csv', '02_nav_history.csv', '03_aum_by_fund_house.csv', '04_monthly_sip_inflows.csv', '05_category_inflows.csv', '06_industry_folio_count.csv', '07_scheme_performance.csv', '08_investor_transactions.csv', '09_portfolio_holdings.csv', '10_benchmark_indices.csv', 'Axis_Bluechip.csv', 'HDFC_Top100.csv', 'ICICI_Bluechip.csv', 'Kotak_Bluechip.csv', 'Nippon_LargeCap.csv', 'SBI_Bluechip.csv']


In [3]:
import pandas as pd

benchmark = pd.read_csv('../data/raw/10_benchmark_indices.csv', parse_dates=['date'])
nifty50 = benchmark[benchmark['index_name'] == 'NIFTY50'].copy()
nifty50['month'] = nifty50['date'].dt.to_period('M').astype(str)

nifty50_monthly = nifty50.groupby('month')['close_value'].mean().reset_index()
nifty50_monthly.to_csv('nifty50_monthly.csv', index=False)
print(nifty50_monthly.shape)
nifty50_monthly.head()

(53, 2)


,month,close_value
0,2022-01,18167.482381
1,2022-02,18802.853500
2,2022-03,19088.483043
3,2022-04,20186.594286
4,2022-05,19530.993636


In [4]:
import os
print(os.getcwd())
print(os.path.exists('nifty50_monthly.csv'))

C:\Users\raghvendr shahi\Desktop\Mutual_Fund_Analytics\notebooks
True


In [1]:
import os
print(os.listdir('.'))

['.ipynb_checkpoints', 'alpha_beta.csv', 'benchmark_comparison_chart.png', 'fund_scorecard.csv', 'nifty50_monthly.csv', 'Performance_Analytics.ipynb', 'run_analytics.py']


In [2]:
import pandas as pd
df = pd.read_csv('../data/raw/09_portfolio_holdings.csv')
print(df.columns.tolist())
print(df.head(5))

['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct', 'market_value_cr', 'current_price_inr', 'portfolio_date']
   amfi_code stock_symbol                stock_name       sector  weight_pct  \
0     119551    POWERGRID    Power Grid Corporation    Utilities       13.85   
1     119551     HDFCBANK             HDFC Bank Ltd      Banking       11.19   
2     119551       GRASIM     Grasim Industries Ltd  Diversified        9.90   
3     119551      DRREDDY  Dr. Reddy's Laboratories       Pharma        4.76   
4     119551   ASIANPAINT          Asian Paints Ltd       Paints       10.25   

   market_value_cr  current_price_inr portfolio_date  
0           737.09            6011.08     2025-12-31  
1            88.97            1074.65     2025-12-31  
2           208.45            5964.59     2025-12-31  
3           161.32            3748.82     2025-12-31  
4           725.90            1321.45     2025-12-31  


In [3]:
for fund_name, code in KEY_FUNDS.items():
    excess_returns = daily_returns[code] - daily_rf
    rolling_sharpe = (excess_returns.rolling(90).mean() / excess_returns.rolling(90).std()) * np.sqrt(252)
    print(fund_name, "- std dev of rolling Sharpe:", rolling_sharpe.std())

NameError: name 'KEY_FUNDS' is not defined